In [11]:
import json

import numpy as np

In [2]:
image_dir = "E:/Research/simplify-me/simplify_me_dataset/images"
meta_path = "E:/Research/simplify-me/simplify_me_dataset/meta.json"
environment = 'local'

REGISTRY = {
    'local': {
        'image_dir': "E:/Research/simplify-me/simplify_me_dataset/images",
        'meta_path': "E:/Research/simplify-me/simplify_me_dataset/meta.json",
        'environment': 'local',
    },
    'cc': {
        'image_dir': "/home/pranon/scratch/def-tahmedge/simplify-me-dataset/simplify-me/compressed_images",
        'meta_path': "E:/Research/simplify-me/simplify_me_dataset/meta.json",
        'environment': 'cc',
    }
}

In [3]:
def load_data():
    with open(meta_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [18]:
def get_and_preprocess_data(data, split, env, ds_size):
    processed_data = []

    data = data[split]

    data = sorted(
        data,
        key=lambda x: float(x['dpo']['sle_delta']),
        reverse=True
    )

    if ds_size != -1:
        data = data[:ds_size]
    print(split, env, len(data), data[-1]['dpo']['sle_delta'])

    benchmark_count = {}
    for item in data:
        if item['benchmark'] not in benchmark_count:
            benchmark_count[item['benchmark']] = {'data': 0, 'sle': []}

        benchmark_count[item['benchmark']]['data'] += 1
        benchmark_count[item['benchmark']]['sle'].append(item['dpo']['sle_delta'])

        processed_data.append({
            'id': str(item['id']),
            'benchmark': item['benchmark'],
            'input': "<image>",
            'images': [f"{REGISTRY[env]['image_dir']}/{item['benchmark']}/{item['image']['file_name']}"],
            'instruction': 'Describe the image in simple terms',
            'chosen': item['dpo']['accepted']['caption'],
            'rejected': item['dpo']['rejected']['caption'],
        })

    print([(x, benchmark_count[x]['data']) for x in benchmark_count])
    print('average', [(x, np.average(benchmark_count[x]['sle'])) for x in benchmark_count])
    print('min', [(x, np.min(benchmark_count[x]['sle'])) for x in benchmark_count])
    print('max', [(x, np.max(benchmark_count[x]['sle'])) for x in benchmark_count])

    filename = f'../data/{split}-{env}{'' if split != 'train' else '-full' if ds_size == -1 else '-'+str(ds_size)}.json'

    # with open(filename, 'w', encoding='utf-8') as fp:
    #     json.dump(processed_data, fp, indent=4, ensure_ascii=False)

    return filename

In [19]:
def get_dataset_info(env, split, filename, ds_size):
    return {
        f'sm-{env}-{split}{'' if split != 'train' else '-full' if ds_size == -1 else '-'+str(ds_size)}': {
            "file_name": filename,
            "ranking": True,
            "columns": {
                "prompt": "instruction",
                "query": "input",
                "chosen": "chosen",
                "rejected": "rejected",
                "images": "images"
            }
        },
    }

In [21]:
raw_data = load_data()

dataset_info = {}
max_size = [-1, 10000, 5000, 1000]

for key in REGISTRY.keys():
    # for sz in max_size:
        # fn = get_and_preprocess_data(raw_data, 'train', key, sz)
        # ds_info = get_dataset_info(key, 'train', fn, sz)
        #
        # dataset_info.update(ds_info)

    fn = get_and_preprocess_data(raw_data, 'train', key, 10000)
    break
    # ds_info = get_dataset_info(key, 'test', fn, 5000)
    # dataset_info.update(ds_info)

# with open('../data/dataset_info.json', 'w', encoding='utf-8') as f:
#     json.dump(dataset_info, f, indent=4, ensure_ascii=False)

train local 10000 3.366675615310669
[('viswiz_train', 1917), ('flickr30k_val', 2444), ('coco_2014_train', 2412), ('viswiz_val', 647), ('coco_2014_val', 1134), ('textcaps_train', 1227), ('textcaps_val', 219)]
average [('viswiz_train', np.float64(3.786613859316887)), ('flickr30k_val', np.float64(3.82624490098165)), ('coco_2014_train', np.float64(3.687493583982869)), ('viswiz_val', np.float64(3.784729418018041)), ('coco_2014_val', np.float64(3.686615641104021)), ('textcaps_train', np.float64(3.6764857303427068)), ('textcaps_val', np.float64(3.6809187926539138))]
min [('viswiz_train', np.float64(3.366692066192627)), ('flickr30k_val', np.float64(3.366675615310669)), ('coco_2014_train', np.float64(3.3667865991592407)), ('viswiz_val', np.float64(3.367195963859558)), ('coco_2014_val', np.float64(3.3671607971191406)), ('textcaps_train', np.float64(3.3666832447052)), ('textcaps_val', np.float64(3.3671857565641403))]
max [('viswiz_train', np.float64(5.418790102005005)), ('flickr30k_val', np.float